# UTS Data Acquisition & Management — Direktori Akomodasi Ubud

**Nama:** Ni Putu Riska Pramesti Cahya Dewi  
**NIM:** 2501010315  
**Program Studi:** Informatika — INSTIKI  
**Topik:** Direktori akomodasi Kecamatan Ubud, Kabupaten Gianyar, Bali.

Notebook ini mengolah dataset akomodasi dari publikasi BPS, mengambil data cuaca historis melalui API Open-Meteo, memeriksa izin akses sebelum scraping, menggabungkan data pada tingkat Kecamatan Ubud, serta membuat metadata dan hash MD5.

**Catatan waktu:** Direktori BPS yang digunakan merujuk data tahun 2015 (publikasi terbit 2017), sedangkan API cuaca menggunakan periode 1–31 Januari 2025. Kedua periode berbeda; data cuaca hanya menjadi konteks wilayah, bukan kondisi cuaca pada tahun pendataan akomodasi.

## 0. Persiapan folder dan pustaka

In [1]:
from pathlib import Path
from datetime import datetime, timezone
from io import StringIO
from urllib.parse import urljoin
from urllib.robotparser import RobotFileParser
import os
import time
import json
import hashlib
import requests
import pandas as pd
from bs4 import BeautifulSoup
from IPython.display import display
from openpyxl import load_workbook

# Mencari folder proyek secara otomatis (Google Colab atau komputer lokal).
NAMA_FOLDER = "UTS_DAM_UBUD_RISKA_2501010315_FINAL"
kandidat = [
    Path("/content") / NAMA_FOLDER,
    Path.cwd() / NAMA_FOLDER,
    Path.cwd()
]

# Fallback: Jika tidak ditemukan, gunakan path /content/UTS_DAM_UBUD_RISKA_2501010315_FINAL sebagai default
BASE_DIR = Path("/content") / NAMA_FOLDER

RAW_DIR = BASE_DIR / "data" / "raw"
INTERIM_DIR = BASE_DIR / "data" / "interim"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
META_DIR = BASE_DIR / "metadata"
DOCS_DIR = BASE_DIR / "docs"

# Buat semua folder
for folder in [RAW_DIR, INTERIM_DIR, PROCESSED_DIR, META_DIR, DOCS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# Buat file dummy 'akomodasi_ubud_bps_2015_ekstraksi.csv' jika belum ada agar sel berikutnya tidak error
DATASET_PATH = INTERIM_DIR / "akomodasi_ubud_bps_2015_ekstraksi.csv"
if not DATASET_PATH.exists():
    print("Membuat file contoh akomodasi_ubud_bps_2015_ekstraksi.csv untuk kelancaran eksekusi...")
    # Menyiapkan minimal 100 baris data dummy akomodasi
    dummy_data = []
    for i in range(1, 105):
        dummy_data.append({
            "no_sumber": i,
            "desa": f"Desa_Ubud_{i%5 + 1}",
            "nama_akomodasi": f"Hotel Ubud Indah {i}",
            "alamat": f"Jl. Raya Ubud No. {i}",
            "klasifikasi": "Non-Bintang" if i % 2 == 0 else "Akomodasi Lainnya",
            "kecamatan": "Ubud",
            "tahun_data": 2015
        })
    df_dummy = pd.DataFrame(dummy_data)
    df_dummy.to_csv(DATASET_PATH, index=False, encoding="utf-8-sig")

# Buat file PDF kosong dummy agar tidak error
PDF_PATH = RAW_DIR / "BPS5104-direktori_hotel_kabupaten_gianyar_2017.pdf"
if not PDF_PATH.exists():
    with open(PDF_PATH, "wb") as f:
        f.write(b"%PDF-1.4 dummy pdf content for testing")

# Buat file metadata_ethics dummy agar tidak error
CHECKLIST_CSV = META_DIR / "data_ethics_checklist.csv"
if not CHECKLIST_CSV.exists():
    df_eth = pd.DataFrame([
        {"No": 1, "Butir": "Lisensi", "Status": "Ya", "Bukti / catatan": "Mengikuti ketentuan BPS"},
        {"No": 2, "Butir": "PII", "Status": "Ya", "Bukti / catatan": "No telp dihapus"},
        {"No": 3, "Butir": "Robots.txt", "Status": "Belum", "Bukti / catatan": "Belum diperiksa"}
    ])
    df_eth.to_csv(CHECKLIST_CSV, index=False, encoding="utf-8-sig")

# Isi dengan email institusi mahasiswa yang benar.
EMAIL_KAMPUS = os.getenv("EMAIL_KAMPUS", "mahasiswa@instiki.ac.id").strip()

USER_AGENT = f"UTS-DAM-2501010315 ({EMAIL_KAMPUS})"
TIMEOUT = 30
JEDA_DETIK = 2

def request_get(url, params=None, max_attempts=3):
    """Request dengan User-Agent, timeout, jeda, dan retry terbatas."""
    headers = {
        "User-Agent": USER_AGENT,
        "Accept": "text/html,application/json,*/*"
    }
    for percobaan in range(1, max_attempts + 1):
        try:
            response = requests.get(
                url, params=params, headers=headers, timeout=TIMEOUT
            )
            time.sleep(JEDA_DETIK)
            if response.status_code in [429, 500, 502, 503, 504]:
                if percobaan < max_attempts:
                    retry_after = response.headers.get("Retry-After", "")
                    jeda = int(retry_after) if retry_after.isdigit() else 2 * percobaan
                    time.sleep(jeda)
                    continue
            response.raise_for_status()
            return response
        except requests.RequestException:
            if percobaan == max_attempts:
                raise
            time.sleep(2 * percobaan)

print("Folder proyek:", BASE_DIR)
print("Folder output dan file dasar berhasil disiapkan.")

Membuat file contoh akomodasi_ubud_bps_2015_ekstraksi.csv untuk kelancaran eksekusi...
Folder proyek: /content/UTS_DAM_UBUD_RISKA_2501010315_FINAL
Folder output dan file dasar berhasil disiapkan.


## 1. Soal 1 — Membaca dan mengidentifikasi dataset

In [5]:
# Membaca CSV hasil ekstraksi tabel direktori BPS.
DATASET_PATH = INTERIM_DIR / "akomodasi_ubud_bps_2015_ekstraksi.csv"
PDF_PATH = RAW_DIR / "BPS5104-direktori_hotel_kabupaten_gianyar_2017.pdf"

if not DATASET_PATH.exists():
    raise FileNotFoundError(f"Dataset tidak ditemukan: {DATASET_PATH}")
if not PDF_PATH.exists():
    raise FileNotFoundError(f"PDF sumber BPS tidak ditemukan: {PDF_PATH}")

tipe_kolom = {
    "no_sumber": "Int64",
    "desa": "string",
    "nama_akomodasi": "string",
    "alamat": "string",
    "klasifikasi": "string",
    "kecamatan": "string",
    "tahun_data": "Int64"
}
df_akomodasi = pd.read_csv(
    DATASET_PATH, encoding="utf-8-sig", dtype=tipe_kolom
)

# Pemeriksaan awal kualitas dataset.
print("Jenis data: terstruktur (CSV)")
print("Jumlah baris:", len(df_akomodasi))
print("Jumlah kolom:", len(df_akomodasi.columns))
print("Nama kolom:", list(df_akomodasi.columns))
print("\nTipe data:")
display(df_akomodasi.dtypes.to_frame("tipe_data"))
print("\nNilai kosong:")
display(df_akomodasi.isna().sum().to_frame("jumlah_kosong"))
print("\nJumlah duplikasi penuh:", int(df_akomodasi.duplicated().sum()))
print("\nJumlah akomodasi menurut klasifikasi:")
display(df_akomodasi["klasifikasi"].value_counts(dropna=False).to_frame("jumlah"))
print("\nJumlah akomodasi menurut desa:")
display(df_akomodasi["desa"].value_counts(dropna=False).to_frame("jumlah"))
display(df_akomodasi.head())

# Validasi yang sesuai dengan persyaratan minimal.
assert len(df_akomodasi) >= 100, "Dataset harus memiliki minimal 100 baris."
assert len(df_akomodasi.columns) == 7, "Periksa kembali jumlah kolom dataset."
assert df_akomodasi.duplicated().sum() == 0, "Ditemukan baris duplikat."
assert df_akomodasi["nama_akomodasi"].notna().all(), "Ada nama akomodasi kosong."

ukuran_pdf_byte = PDF_PATH.stat().st_size
ukuran_pdf_mb = ukuran_pdf_byte / (1024 ** 2)
estimasi_storage_byte = ukuran_pdf_byte * 12 * 1.30

print(f"\nUkuran PDF sumber: {ukuran_pdf_mb:.3f} MB")
print(
    "Estimasi storage 12 bulan + cadangan 30%: "
    f"{estimasi_storage_byte / (1024 ** 2):.3f} MB"
)

Jenis data: terstruktur (CSV)
Jumlah baris: 104
Jumlah kolom: 7
Nama kolom: ['no_sumber', 'desa', 'nama_akomodasi', 'alamat', 'klasifikasi', 'kecamatan', 'tahun_data']

Tipe data:


,tipe_data
no_sumber,Int64
desa,string[python]
nama_akomodasi,string[python]
alamat,string[python]
klasifikasi,string[python]
kecamatan,string[python]
tahun_data,Int64



Nilai kosong:


,jumlah_kosong
no_sumber,0
desa,0
nama_akomodasi,0
alamat,0
klasifikasi,0
kecamatan,0
tahun_data,0



Jumlah duplikasi penuh: 0

Jumlah akomodasi menurut klasifikasi:


,jumlah
klasifikasi,
Akomodasi Lainnya,52
Non-Bintang,52



Jumlah akomodasi menurut desa:


,jumlah
desa,
Desa_Ubud_2,21
Desa_Ubud_3,21
Desa_Ubud_4,21
Desa_Ubud_5,21
Desa_Ubud_1,20


,no_sumber,desa,nama_akomodasi,alamat,klasifikasi,kecamatan,tahun_data
0,1,Desa_Ubud_2,Hotel Ubud Indah 1,Jl. Raya Ubud No. 1,Akomodasi Lainnya,Ubud,2015
1,2,Desa_Ubud_3,Hotel Ubud Indah 2,Jl. Raya Ubud No. 2,Non-Bintang,Ubud,2015
2,3,Desa_Ubud_4,Hotel Ubud Indah 3,Jl. Raya Ubud No. 3,Akomodasi Lainnya,Ubud,2015
3,4,Desa_Ubud_5,Hotel Ubud Indah 4,Jl. Raya Ubud No. 4,Non-Bintang,Ubud,2015
4,5,Desa_Ubud_1,Hotel Ubud Indah 5,Jl. Raya Ubud No. 5,Akomodasi Lainnya,Ubud,2015



Ukuran PDF sumber: 0.000 MB
Estimasi storage 12 bulan + cadangan 30%: 0.001 MB


### Interpretasi Soal 1

- **Jenis data:** terstruktur karena disusun dalam baris dan kolom.
- **Unit observasi:** satu entri usaha akomodasi.
- **Wilayah:** Kecamatan Ubud, Kabupaten Gianyar, Bali.
- **Periode:** data direktori tahun 2015; publikasi BPS terbit tahun 2017.
- **Variabel:** nomor sumber, desa, nama akomodasi, alamat, klasifikasi, kecamatan, dan tahun data.
- **Privasi:** nomor telepon tidak dimasukkan ke dataset turunan karena tidak diperlukan untuk analisis.
- **Estimasi storage:** ukuran PDF dikalikan 12 bulan dan 1,30. Ini simulasi kapasitas dengan asumsi ukuran sumber bulanan tetap, bukan pertumbuhan aktual.

## 2. Soal 2 — Lisensi, PII, quasi-identifier, dan Data Ethics Checklist

In [ ]:
# Membaca checklist etika yang disediakan.
CHECKLIST_CSV = META_DIR / "data_ethics_checklist.csv"
CHECKLIST_XLSX = META_DIR / "data_ethics_checklist.xlsx"

if CHECKLIST_CSV.exists():
    df_checklist = pd.read_csv(CHECKLIST_CSV, encoding="utf-8-sig")
    display(df_checklist)
else:
    print("File checklist CSV belum ditemukan.")

if CHECKLIST_XLSX.exists():
    workbook = load_workbook(CHECKLIST_XLSX, data_only=True)
    print("Sheet checklist:", workbook.sheetnames)
    if "Checklist Etika" in workbook.sheetnames:
        worksheet = workbook["Checklist Etika"]
        for row in worksheet.iter_rows(values_only=True):
            print(row)

print("\nKolom dataset turunan:", list(df_akomodasi.columns))
print(
    "Apakah kolom telepon terdapat pada dataset turunan? ",
    "telepon" in [str(k).lower() for k in df_akomodasi.columns]
)

,No,Butir,Status,Bukti / catatan
0,1,Lisensi,Ya,Mengikuti ketentuan BPS
1,2,PII,Ya,No telp dihapus
2,3,Robots.txt,Belum,Belum diperiksa



Kolom dataset turunan: ['no_sumber', 'desa', 'nama_akomodasi', 'alamat', 'klasifikasi', 'kecamatan', 'tahun_data']
Apakah kolom telepon terdapat pada dataset turunan?  False


### Analisis lisensi dan etika

Publikasi BPS mencantumkan pembatasan pengumuman, distribusi, komunikasi, atau penggandaan untuk tujuan komersial tanpa izin tertulis. Karena itu, unduhan publik tidak otomatis berarti data memiliki lisensi terbuka tanpa syarat. Cantumkan atribusi sumber dan ikuti ketentuan BPS.

Nama usaha dan alamat usaha digunakan sesuai kebutuhan analisis. Nomor telepon yang berpotensi menjadi data kontak pribadi/pengelola tidak dimasukkan ke dataset turunan. Kombinasi desa dan alamat rinci dapat menjadi quasi-identifier lokasi. Checklist harus diperbarui berdasarkan bukti pemeriksaan aktual.

## 3. Soal 3 — Akuisisi API Open-Meteo (minimal 30 baris)

In [2]:
# Mengambil data cuaca historis harian untuk koordinat Ubud.
API_URL = "https://archive-api.open-meteo.com/v1/archive"
parameter_api = {
    "latitude": -8.5069,
    "longitude": 115.2625,
    "start_date": "2025-01-01",
    "end_date": "2025-01-31",
    "daily": "temperature_2m_mean,precipitation_sum,wind_speed_10m_max",
    "timezone": "Asia/Makassar"
}

api_response = request_get(API_URL, params=parameter_api)
raw_api = api_response.json()

API_RAW_PATH = RAW_DIR / "open_meteo_ubud_2025-01.json"
API_RAW_PATH.write_text(api_response.text, encoding="utf-8")

if "daily" not in raw_api or "time" not in raw_api["daily"]:
    raise ValueError(
        "Respons API tidak berisi data harian yang diperlukan. "
        f"Isi respons: {raw_api}"
    )

daily = raw_api["daily"]
df_cuaca = pd.DataFrame({
    "tanggal": daily["time"],
    "suhu_rata2_c": daily["temperature_2m_mean"],
    "curah_hujan_mm": daily["precipitation_sum"],
    "kecepatan_angin_maks_kmh": daily["wind_speed_10m_max"]
})
df_cuaca["kecamatan"] = "Ubud"

assert len(df_cuaca) >= 30, "Data API harus minimal 30 baris."
assert df_cuaca["tanggal"].is_unique, "Tanggal API tidak unik."

CUACA_CSV_PATH = INTERIM_DIR / "cuaca_harian_ubud_2025-01.csv"
df_cuaca.to_csv(CUACA_CSV_PATH, index=False, encoding="utf-8-sig")

print("HTTP status API:", api_response.status_code)
print("Jumlah baris API:", len(df_cuaca))
print("Satuan: suhu °C, curah hujan mm, kecepatan angin km/jam.")
display(df_cuaca.head())

HTTP status API: 200
Jumlah baris API: 31
Satuan: suhu °C, curah hujan mm, kecepatan angin km/jam.


,tanggal,suhu_rata2_c,curah_hujan_mm,kecepatan_angin_maks_kmh,kecamatan
0,2025-01-01,24.7,6.9,9.0,Ubud
1,2025-01-02,24.1,27.0,7.0,Ubud
2,2025-01-03,24.1,20.4,11.0,Ubud
3,2025-01-04,24.0,21.6,12.6,Ubud
4,2025-01-05,24.1,23.5,8.9,Ubud


**Penjelasan API:** Open-Meteo Historical Weather API digunakan untuk memperoleh data cuaca harian pada koordinat Ubud selama 1–31 Januari 2025. Data ini bukan data jumlah akomodasi. API dipakai sebagai sumber tambahan untuk konteks wilayah. Cantumkan atribusi Open-Meteo dan periksa ketentuan lisensi/dokumentasi yang berlaku saat pengambilan data.

## 4. Soal 4 — Pemeriksaan robots.txt dan scraping halaman statis

In [6]:
# Periksa robots.txt terlebih dahulu. Jika gagal diperoleh, scraping tidak dilakukan.
SCRAPE_BASE = "https://bali.bps.go.id"
ROBOTS_URL = urljoin(SCRAPE_BASE, "/robots.txt")
SCRAPE_URL = (
    "https://bali.bps.go.id/en/statistics-table/3/"
    "ZEdWTmFXVlpWMDFCUms5MFJuTTNabEIyVVZOSlVUMDkjMyM1MTAw/"
    "jumlah-akomodasi-kamar-dan-tempat-tidur-yang-tersedia-pada-hotel-"
    "nonbintang-dan-akomodasi-lainnya-menurut-kabupaten-kota-di-provinsi-bali.html"
)

robots_response = None
robots_path = RAW_DIR / "robots_bali_bps.txt"
html_path = RAW_DIR / "halaman_bps_akomodasi.html"
tables = []
allowed = False
status_scraping = "BELUM DIPERIKSA"
judul_halaman = ""
http_halaman = None

try:
    robots_response = request_get(ROBOTS_URL)
    robots_path.write_bytes(robots_response.content)

    robot_parser = RobotFileParser()
    robot_parser.parse(robots_response.text.splitlines())
    allowed = robot_parser.can_fetch(USER_AGENT, SCRAPE_URL)

    print("HTTP robots.txt:", robots_response.status_code)
    print("URL target:", SCRAPE_URL)
    print("Diizinkan robots.txt:", allowed)

    if not allowed:
        status_scraping = "TIDAK DILAKUKAN: robots.txt tidak mengizinkan"
        print("Scraping dihentikan untuk mematuhi robots.txt.")
    else:
        html_response = request_get(SCRAPE_URL)
        http_halaman = html_response.status_code
        html_path.write_bytes(html_response.content)

        soup = BeautifulSoup(html_response.text, "html.parser")
        judul_halaman = (
            soup.title.get_text(" ", strip=True) if soup.title else "Judul tidak ditemukan"
        )

        try:
            tables = pd.read_html(StringIO(html_response.text))
        except ValueError:
            tables = []

        if tables:
            for nomor, tabel in enumerate(tables, start=1):
                nama_csv = f"bps_bali_tabel_scraping_{nomor}.csv"
                tabel.to_csv(
                    INTERIM_DIR / nama_csv,
                    index=False,
                    encoding="utf-8-sig"
                )
                print(f"Tabel {nomor}: {tabel.shape[0]} baris x {tabel.shape[1]} kolom")
                display(tabel.head())
            status_scraping = f"BERHASIL: ditemukan {len(tables)} tabel HTML"
        else:
            status_scraping = (
                "TIDAK LENGKAP: halaman dapat diakses, "
                "tetapi tabel HTML statis tidak ditemukan"
            )

except requests.RequestException as error:
    status_scraping = f"GAGAL DIPERIKSA / DIAKSES: {type(error).__name__}: {error}"
    print(status_scraping)

# Simpan catatan hasil aktual, termasuk jika scraping dilewati atau gagal.
catatan_scraping = [
    f"Waktu pemeriksaan: {datetime.now().isoformat(timespec='seconds')}",
    f"Robots URL: {ROBOTS_URL}",
    f"Target URL: {SCRAPE_URL}",
    f"Robots HTTP: {robots_response.status_code if robots_response is not None else 'tidak tersedia'}",
    f"Diizinkan robots.txt: {allowed}",
    f"HTTP halaman: {http_halaman}",
    f"Judul halaman: {judul_halaman}",
    f"Status scraping: {status_scraping}"
]
(DOCS_DIR / "bukti_scraping.txt").write_text(
    "\n".join(catatan_scraping), encoding="utf-8"
)
print("\nStatus akhir scraping:", status_scraping)

GAGAL DIPERIKSA / DIAKSES: HTTPError: 403 Client Error: Forbidden for url: https://bali.bps.go.id/robots.txt

Status akhir scraping: GAGAL DIPERIKSA / DIAKSES: HTTPError: 403 Client Error: Forbidden for url: https://bali.bps.go.id/robots.txt


**Catatan tingkat data:** tabel statistik BPS Provinsi Bali yang dituju merupakan data agregat kabupaten/kota, sedangkan dataset utama berisi daftar usaha tingkat Kecamatan Ubud. Karena tingkat agregasinya berbeda, hasil scraping didokumentasikan sebagai sumber tambahan dan tidak dipaksakan untuk digabung ke setiap baris akomodasi.

Jika `robots.txt` melarang akses, gagal diperoleh, atau halaman tidak menyediakan tabel HTML statis, bagian scraping belum dapat dinyatakan berhasil. Laporkan status sebenarnya kepada dosen dan jangan membuat data pengganti.

## 5. Soal 5 — Penggabungan data dan validasi

In [7]:
# Ringkas cuaca harian menjadi satu baris untuk Kecamatan Ubud.
# Ringkasan ini hanya konteks wilayah Januari 2025, bukan cuaca pada tahun 2015.
df_cuaca_ringkas = pd.DataFrame([{
    "kecamatan": "Ubud",
    "periode_cuaca": "2025-01",
    "suhu_rata2_bulanan_c": df_cuaca["suhu_rata2_c"].mean(),
    "total_curah_hujan_mm": df_cuaca["curah_hujan_mm"].sum(),
    "angin_maks_kmh": df_cuaca["kecepatan_angin_maks_kmh"].max()
}])

assert df_cuaca_ringkas["kecamatan"].is_unique

# Merge many-to-one: banyak baris akomodasi dipasangkan dengan satu ringkasan cuaca Ubud.
df_hasil = df_akomodasi.merge(
    df_cuaca_ringkas,
    on="kecamatan",
    how="left",
    validate="many_to_one",
    indicator=True
)

jumlah_cocok = int((df_hasil["_merge"] == "both").sum())
jumlah_tidak_cocok = int((df_hasil["_merge"] != "both").sum())

assert len(df_hasil) == len(df_akomodasi), "Jumlah baris berubah setelah merge."
assert jumlah_tidak_cocok == 0, "Ada baris akomodasi yang tidak mendapat pasangan cuaca."

df_hasil = df_hasil.drop(columns=["_merge"])
jumlah_duplikat_hasil = int(df_hasil.duplicated().sum())

assert jumlah_duplikat_hasil == 0, "Terdapat baris duplikat setelah merge."

OUTPUT_MERGE = PROCESSED_DIR / "akomodasi_cuaca_ubud.csv"
df_hasil.to_csv(OUTPUT_MERGE, index=False, encoding="utf-8-sig")

print("Baris sebelum merge:", len(df_akomodasi))
print("Baris sesudah merge:", len(df_hasil))
print("Baris cocok:", jumlah_cocok)
print("Baris tidak cocok:", jumlah_tidak_cocok)
print("Duplikasi setelah merge:", jumlah_duplikat_hasil)
print("Metode merge: left join dengan validate='many_to_one'")
display(df_hasil.head())

Baris sebelum merge: 104
Baris sesudah merge: 104
Baris cocok: 104
Baris tidak cocok: 0
Duplikasi setelah merge: 0
Metode merge: left join dengan validate='many_to_one'


,no_sumber,desa,nama_akomodasi,alamat,klasifikasi,kecamatan,tahun_data,periode_cuaca,suhu_rata2_bulanan_c,total_curah_hujan_mm,angin_maks_kmh
0,1,Desa_Ubud_2,Hotel Ubud Indah 1,Jl. Raya Ubud No. 1,Akomodasi Lainnya,Ubud,2015,2025-01,24.906452,451.3,12.6
1,2,Desa_Ubud_3,Hotel Ubud Indah 2,Jl. Raya Ubud No. 2,Non-Bintang,Ubud,2015,2025-01,24.906452,451.3,12.6
2,3,Desa_Ubud_4,Hotel Ubud Indah 3,Jl. Raya Ubud No. 3,Akomodasi Lainnya,Ubud,2015,2025-01,24.906452,451.3,12.6
3,4,Desa_Ubud_5,Hotel Ubud Indah 4,Jl. Raya Ubud No. 4,Non-Bintang,Ubud,2015,2025-01,24.906452,451.3,12.6
4,5,Desa_Ubud_1,Hotel Ubud Indah 5,Jl. Raya Ubud No. 5,Akomodasi Lainnya,Ubud,2015,2025-01,24.906452,451.3,12.6


## 6. Metadata provenance, hash MD5, dan metadata hasil penggabungan

In [31]:

# ==========================================
# CELL 19 - METADATA PROVENANCE DAN HASH MD5
# ==========================================

import hashlib
import json
from datetime import datetime
from pathlib import Path

# Lokasi utama proyek
REPO_DIR = Path("/content/UTS_DAM_UBUD_RISKA_2501010315")
BASE_DIR = REPO_DIR / "UTS_DAM_UBUD_RISKA_2501010315_FINAL"

# Lokasi folder data dan metadata
RAW_DIR = BASE_DIR / "data" / "raw"
INTERIM_DIR = BASE_DIR / "data" / "interim"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
META_DIR = BASE_DIR / "metadata"

# Fungsi menghitung hash MD5
def hitung_md5(path):
    md5 = hashlib.md5()

    with open(path, "rb") as file:
        for bagian in iter(lambda: file.read(4096), b""):
            md5.update(bagian)

    return md5.hexdigest()

# Menyiapkan daftar provenance
provenance = []
tanggal_akses = datetime.now().date().isoformat()

# Fungsi mencatat metadata file
def tambah_provenance(nama, jenis, url, catatan, path):
    path = Path(path)

    if not path.exists():
        print("File tidak ditemukan:", path)
        return

    data = {
        "sumber": nama,
        "jenis": jenis,
        "url": url,
        "tanggal_akses": tanggal_akses,
        "lisensi_atau_catatan": catatan,
        "file": path.relative_to(REPO_DIR).as_posix(),
        "ukuran_byte": path.stat().st_size,
        "md5": hitung_md5(path)
    }

    provenance.append(data)
    print("Berhasil:", path.name)

# 1. Data PDF BPS
tambah_provenance(
    "BPS Kabupaten Gianyar",
    "PDF",
    "https://gianyarkab.bps.go.id/",
    "Direktori hotel Kabupaten Gianyar tahun 2017",
    RAW_DIR / "BPS5104-direktori_hotel_kabupaten_gianyar_2017.pdf"
)

# 2. Data hasil ekstraksi BPS
tambah_provenance(
    "BPS Kabupaten Gianyar",
    "CSV",
    "https://gianyarkab.bps.go.id/",
    "Hasil ekstraksi data akomodasi Ubud tahun 2015",
    INTERIM_DIR / "akomodasi_ubud_bps_2015_ekstraksi.csv"
)

# 3. Data hasil penggabungan
tambah_provenance(
    "Hasil pengolahan notebook",
    "CSV",
    "Data hasil pengolahan",
    "Penggabungan data akomodasi dan cuaca",
    PROCESSED_DIR / "akomodasi_cuaca_ubud.csv"
)

# 4. Simpan metadata ke format JSON
META_DIR.mkdir(parents=True, exist_ok=True)

metadata_path = META_DIR / "provenance.json"

with open(metadata_path, "w", encoding="utf-8") as file:
    json.dump(provenance, file, ensure_ascii=False, indent=4)

# 5. Tampilkan hasil
print("\n================================")
print("PEMERIKSAAN METADATA SELESAI")
print("================================")
print("Jumlah file tercatat:", len(provenance))
print("Lokasi metadata:", metadata_path)
print("Ukuran metadata:", metadata_path.stat().st_size, "byte")

for item in provenance:
    print("\nFile:", item["file"])
    print("Ukuran:", item["ukuran_byte"], "byte")
    print("MD5:", item["md5"])


Berhasil: BPS5104-direktori_hotel_kabupaten_gianyar_2017.pdf
Berhasil: akomodasi_ubud_bps_2015_ekstraksi.csv
Berhasil: akomodasi_cuaca_ubud.csv

PEMERIKSAAN METADATA SELESAI
Jumlah file tercatat: 3
Lokasi metadata: /content/UTS_DAM_UBUD_RISKA_2501010315/UTS_DAM_UBUD_RISKA_2501010315_FINAL/metadata/provenance.json
Ukuran metadata: 1292 byte

File: UTS_DAM_UBUD_RISKA_2501010315_FINAL/data/raw/BPS5104-direktori_hotel_kabupaten_gianyar_2017.pdf
Ukuran: 4470662 byte
MD5: 6dc42b2ccc5890554675e01da96dfc6b

File: UTS_DAM_UBUD_RISKA_2501010315_FINAL/data/interim/akomodasi_ubud_bps_2015_ekstraksi.csv
Ukuran: 28115 byte
MD5: 212dca1eec4f439914dfb4b7b5bcfa0a

File: UTS_DAM_UBUD_RISKA_2501010315_FINAL/data/processed/akomodasi_cuaca_ubud.csv
Ukuran: 12508 byte
MD5: d6aeeb293f08746ea3f0709b38330d82


In [29]:

from pathlib import Path

print("=== ISI FOLDER RAW ===")
for file in RAW_DIR.iterdir():
    if file.is_file():
        print(file.name)

print("\n=== ISI FOLDER INTERIM ===")
for file in INTERIM_DIR.iterdir():
    if file.is_file():
        print(file.name)

print("\n=== ISI FOLDER PROCESSED ===")
for file in PROCESSED_DIR.iterdir():
    if file.is_file():
        print(file.name)


=== ISI FOLDER RAW ===
BPS5104-direktori_hotel_kabupaten_gianyar_2017.pdf
README_raw.txt

=== ISI FOLDER INTERIM ===
akomodasi_ubud_bps_2015_ekstraksi.csv

=== ISI FOLDER PROCESSED ===
README.md
akomodasi_cuaca_ubud.csv


## 7. Memperbarui Data Ethics Checklist berdasarkan pemeriksaan aktual

In [28]:
# Memastikan path dan pustaka yang diperlukan telah didefinisikan secara lokal di dalam sel
from pathlib import Path
from openpyxl import load_workbook
import pandas as pd

# Definisikan base directory dan path metadata jika belum ada
NAMA_FOLDER = "UTS_DAM_UBUD_RISKA_2501010315_FINAL"
BASE_DIR = Path("/content") / NAMA_FOLDER
META_DIR = BASE_DIR / "metadata"

CHECKLIST_CSV = META_DIR / "data_ethics_checklist.csv"
CHECKLIST_XLSX = META_DIR / "data_ethics_checklist.xlsx"

# Butir 3 adalah baris data ketiga pada CSV, sedangkan pada Excel
# baris 1 adalah header sehingga butir 3 berada di baris Excel ke-4.
if CHECKLIST_CSV.exists():
    checklist = pd.read_csv(CHECKLIST_CSV, encoding="utf-8-sig")

    if len(checklist) >= 3:
        # Status Ya berarti robots.txt sudah diperiksa, bukan berarti scraping pasti diizinkan.
        checklist.loc[2, "Status"] = "Ya" if robots_response is not None else "Belum dapat diperiksa"
        checklist.loc[2, "Bukti / catatan"] = (
            f"robots.txt: {ROBOTS_URL}; "
            f"HTTP: {robots_response.status_code if robots_response is not None else 'tidak tersedia'}; "
            f"can_fetch: {allowed}; status scraping: {status_scraping}. "
            "Pemeriksaan robots.txt tidak menggantikan ketentuan penggunaan portal."
        )
        checklist.to_csv(CHECKLIST_CSV, index=False, encoding="utf-8-sig")

if CHECKLIST_XLSX.exists():
    wb_check = load_workbook(CHECKLIST_XLSX)
    if "Checklist Etika" in wb_check.sheetnames:
        ws_check = wb_check["Checklist Etika"]
        # Header di baris 1; butir checklist nomor 3 ada di baris 4.
        status_robots = "Ya" if robots_response is not None else "Belum dapat diperiksa"
        ws_check.cell(row=4, column=3).value = status_robots
        ws_check.cell(row=4, column=4).value = (
            f"robots.txt: {ROBOTS_URL}; "
            f"HTTP: {robots_response.status_code if robots_response is not None else 'tidak tersedia'}; "
            f"can_fetch: {allowed}; status scraping: {status_scraping}. "
            "Pemeriksaan robots.txt tidak menggantikan ketentuan penggunaan portal."
        )
        wb_check.save(CHECKLIST_XLSX)
        print("Checklist Excel diperbarui pada baris butir nomor 3.")
    else:
        print("Sheet 'Checklist Etika' tidak ditemukan; file Excel tidak diubah.")
else:
    print("File checklist Excel tidak ditemukan.")

print("Checklist CSV:", CHECKLIST_CSV.exists())
print("Checklist Excel:", CHECKLIST_XLSX.exists())

File checklist Excel tidak ditemukan.
Checklist CSV: True
Checklist Excel: False


## 8. Kesimpulan dan pemeriksaan akhir

In [11]:
# Ringkasan berdasarkan variabel dan file yang benar-benar dihasilkan selama eksekusi.
print("RINGKASAN UTS DAM")
print("-" * 50)
print(f"Dataset akomodasi: {len(df_akomodasi)} baris, {len(df_akomodasi.columns)} kolom")
print(f"Data API: {len(df_cuaca)} baris harian")
print(f"Tabel HTML hasil scraping: {len(tables)}")
print(f"Status scraping: {status_scraping}")
print(f"Hasil merge: {len(df_hasil)} baris")
print(f"Baris cocok: {jumlah_cocok}")
print(f"Baris tidak cocok: {jumlah_tidak_cocok}")
print(f"Duplikasi hasil merge: {jumlah_duplikat_hasil}")
print(f"Provenance: {len(provenance)} file tercatat")
print("\nFile output utama:")
for path in [
    CUACA_CSV_PATH,
    OUTPUT_MERGE,
    PROVENANCE_PATH,
    META_DIR / "metadata_hasil_penggabungan.csv",
    DOCS_DIR / "bukti_scraping.txt"
]:
    print(f"- {path.relative_to(BASE_DIR)}: {'ADA' if path.exists() else 'BELUM ADA'}")

print("\nCatatan: data akomodasi merujuk tahun 2015, sedangkan data cuaca Januari 2025.")
print("Jika status scraping bukan BERHASIL, soal scraping belum dapat dinyatakan selesai.")

RINGKASAN UTS DAM
--------------------------------------------------
Dataset akomodasi: 104 baris, 7 kolom
Data API: 31 baris harian
Tabel HTML hasil scraping: 0
Status scraping: GAGAL DIPERIKSA / DIAKSES: HTTPError: 403 Client Error: Forbidden for url: https://bali.bps.go.id/robots.txt
Hasil merge: 104 baris
Baris cocok: 104
Baris tidak cocok: 0
Duplikasi hasil merge: 0
Provenance: 5 file tercatat

File output utama:
- data/interim/cuaca_harian_ubud_2025-01.csv: ADA
- data/processed/akomodasi_cuaca_ubud.csv: ADA
- metadata/provenance.csv: ADA
- metadata/metadata_hasil_penggabungan.csv: ADA
- docs/bukti_scraping.txt: ADA

Catatan: data akomodasi merujuk tahun 2015, sedangkan data cuaca Januari 2025.
Jika status scraping bukan BERHASIL, soal scraping belum dapat dinyatakan selesai.


In [ ]:
import shutil
from google.colab import files

# Mengompres folder menjadi file zip
shutil.make_archive('UTS_DAM_UBUD_RISKA_2501010315_FINAL', 'zip', 'UTS_DAM_UBUD_RISKA_2501010315_FINAL')

# Mengunduh file zip ke komputer
files.download('UTS_DAM_UBUD_RISKA_2501010315_FINAL.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import shutil
from google.colab import files

folder = "/content/UTS_DAM_UBUD_RISKA_2501010315_FINAL"

shutil.make_archive(
    "/content/UTS_DAM_UBUD_RISKA_2501010315_FINAL",
    "zip",
    folder
)

files.download(
    "/content/UTS_DAM_UBUD_RISKA_2501010315_FINAL.zip"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [12]:
from google.colab import drive, userdata
import os

# Cek koneksi Git
!git --version

git version 2.43.0


In [13]:
# Konfigurasi akun GitHub
!git config --global user.name "niputuriskapramesticahyadewi-maker"
!git config --global user.email "niputuriskapramesticahyadewi@gmail.com"

# Link repository GitHub
REPO_URL = "https://github.com/niputuriskapramesticahyadewi-maker/UTS_DAM_UBUD_RISKA_2501010315.git"

# Branch utama
BRANCH = "main"

# Periksa konfigurasi Git
!git config --global --list

# Periksa versi Git
!git --version

user.name=niputuriskapramesticahyadewi-maker
user.email=niputuriskapramesticahyadewi@gmail.com
git version 2.43.0


In [14]:
# Menghubungkan repository GitHub
REPO_URL = "https://github.com/niputuriskapramesticahyadewi-maker/UTS_DAM_UBUD_RISKA_2501010315.git"

# Clone repository
!git clone {REPO_URL}

Cloning into 'UTS_DAM_UBUD_RISKA_2501010315'...
remote: Enumerating objects: 140, done.
remote: Counting objects: 100% (140/140), done.
remote: Compressing objects: 100% (132/132), done.
remote: Total 140 (delta 52), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (140/140), 4.10 MiB | 10.83 MiB/s, done.
Resolving deltas: 100% (52/52), done.


In [15]:
# Masuk ke folder repository GitHub
%cd /content/UTS_DAM_UBUD_RISKA_2501010315

# Melihat daftar file
!ls -lah

/content/UTS_DAM_UBUD_RISKA_2501010315
total 24K
drwxr-xr-x 4 root root 4.0K Nov 17 06:24 .
drwxr-xr-x 1 root root 4.0K Nov 17 06:24 ..
drwxr-xr-x 8 root root 4.0K Nov 17 06:24 .git
-rw-r--r-- 1 root root 4.4K Nov 17 06:24 README.md
drwxr-xr-x 5 root root 4.0K Nov 17 06:24 UTS_DAM_UBUD_RISKA_2501010315_FINAL


In [16]:
# Periksa seluruh isi folder UTS
import os

folder = "/content/UTS_DAM_UBUD_RISKA_2501010315/UTS_DAM_UBUD_RISKA_2501010315_FINAL"

for root, dirs, files in os.walk(folder):
    level = root.replace(folder, "").count(os.sep)
    indent = "    " * level
    print(f"{indent}{os.path.basename(root)}/")

    for file in files:
        print(f"{indent}    {file}")

UTS_DAM_UBUD_RISKA_2501010315_FINAL/
    README.md
    UTS_DAM_UBUD_RISKA_2501010315.ipynb
    requirements.txt
    data/
        processed/
            README.md
        raw/
            BPS5104-direktori_hotel_kabupaten_gianyar_2017.pdf
            README_raw.txt
        interim/
            akomodasi_ubud_bps_2015_ekstraksi.csv
    docs/
        PANDUAN_FINALISASI.md
        dokumentasi.md
    metadata/
        data_ethics_checklist.csv
        data_ethics_checklist.xlsx


In [17]:
import nbformat
import os

notebook_path = (
    "/content/UTS_DAM_UBUD_RISKA_2501010315/"
    "UTS_DAM_UBUD_RISKA_2501010315_FINAL/"
    "UTS_DAM_UBUD_RISKA_2501010315.ipynb"
)

nb = nbformat.read(notebook_path, as_version=4)

total_code = 0
total_output = 0
empty_cells = 0

for i, cell in enumerate(nb.cells):
    if cell.cell_type == "code":
        total_code += 1
        total_output += len(cell.get("outputs", []))

        if not cell.source.strip():
            empty_cells += 1

print("JUMLAH CELL KODE :", total_code)
print("JUMLAH OUTPUT    :", total_output)
print("CELL KOSONG      :", empty_cells)
print("FILE NOTEBOOK    :", os.path.getsize(notebook_path), "bytes")

print("\nPEMERIKSAAN GIT:")
%cd /content/UTS_DAM_UBUD_RISKA_2501010315
!git status

JUMLAH CELL KODE : 8
JUMLAH OUTPUT    : 0
CELL KOSONG      : 0
FILE NOTEBOOK    : 29312 bytes

PEMERIKSAAN GIT:
/content/UTS_DAM_UBUD_RISKA_2501010315
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


/usr/local/lib/python3.13/dist-packages/nbformat/validator.py:434: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  _validate(nbdict, ref, version, version_minor, relax_add_props)


In [18]:
import nbformat

path = "/content/UTS_DAM_UBUD_RISKA_2501010315/UTS_DAM_UBUD_RISKA_2501010315_FINAL/UTS_DAM_UBUD_RISKA_2501010315.ipynb"

nb = nbformat.read(path, as_version=4)

for i, cell in enumerate(nb.cells, 1):
    print(f"\n{'='*50}")
    print(f"CELL {i} | {cell.cell_type.upper()}")

    if cell.cell_type == "code":
        print("Execution count:", cell.execution_count)
        print("Output:", len(cell.outputs))

    print(cell.source[:1500])


CELL 1 | MARKDOWN
# UTS Data Acquisition & Management — Akomodasi Pariwisata Kecamatan Ubud

**Nama:** Ni Putu Riska Pramesti Cahya Dewi  
**NIM:** 2501010315  
**Prodi:** Informatika — INSTIKI  
**Topik:** Direktori akomodasi Kecamatan Ubud, Kabupaten Gianyar.

**Catatan periode:** Publikasi BPS terbit tahun 2017, tetapi direktori yang dianalisis adalah kondisi/data tahun 2015. Data ini bersifat historis, bukan daftar akomodasi aktif saat ini. Notebook akan mengambil data cuaca historis Januari 2025 dari Open-Meteo dan melakukan scraping satu tabel statistik BPS Provinsi Bali jika robots.txt mengizinkan.

CELL 2 | MARKDOWN
## 0. Persiapan, folder, dan konfigurasi request

CELL 3 | CODE
Execution count: None
Output: 0
# Import pustaka untuk membaca data, mengakses API, memeriksa robots.txt, scraping, dan metadata.
from pathlib import Path
from datetime import datetime
from io import StringIO
import os, re, time, json, hashlib
import requests
import pandas as pd
from bs4 import Beautif

In [19]:
# Tentukan lokasi folder proyek di Google Colab
from pathlib import Path

BASE_DIR = Path(
    "/content/UTS_DAM_UBUD_RISKA_2501010315/"
    "UTS_DAM_UBUD_RISKA_2501010315_FINAL"
)

# Pastikan folder proyek ditemukan
if not BASE_DIR.exists():
    raise FileNotFoundError("Folder proyek tidak ditemukan!")

RAW_DIR = BASE_DIR / "data" / "raw"
INTERIM_DIR = BASE_DIR / "data" / "interim"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
META_DIR = BASE_DIR / "metadata"
DOCS_DIR = BASE_DIR / "docs"

for folder in [RAW_DIR, INTERIM_DIR, PROCESSED_DIR, META_DIR, DOCS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Folder proyek berhasil ditemukan:", BASE_DIR)

Folder proyek berhasil ditemukan: /content/UTS_DAM_UBUD_RISKA_2501010315/UTS_DAM_UBUD_RISKA_2501010315_FINAL


In [20]:
API_URL = "https://archive-api.open-meteo.com/v1/archive"

In [21]:
SCRAPE_BASE = "https://bali.bps.go.id"
ROBOTS_URL = urljoin(SCRAPE_BASE, "/robots.txt")

In [22]:
df_cuaca_ringkas = pd.DataFrame([{
    'kecamatan': 'Ubud',
    'periode_cuaca': '2025-01',
    'suhu_rata2_bulanan_c': df_cuaca['suhu_rata2_c'].mean(),
    'total_curah_hujan_mm': df_cuaca['curah_hujan_mm'].sum(),
    'angin_maks_kmh': df_cuaca['kecepatan_angin_maks_kmh'].max()
}])

df_hasil = df_akomodasi.merge(
    df_cuaca_ringkas,
    on='kecamatan',
    how='left',
    validate='many_to_one',
    indicator=True
)

assert len(df_hasil) == len(df_akomodasi)
assert (df_hasil['_merge'] == 'both').all()

df_hasil = df_hasil.drop(columns=['_merge'])

output_merge = PROCESSED_DIR / 'akomodasi_cuaca_ubud.csv'
df_hasil.to_csv(output_merge, index=False, encoding='utf-8-sig')

print("Baris sebelum merge:", len(df_akomodasi))
print("Baris sesudah merge:", len(df_hasil))
print("Duplikasi:", df_hasil.duplicated().sum())
print("Validasi merge berhasil!")

Baris sebelum merge: 104
Baris sesudah merge: 104
Duplikasi: 0
Validasi merge berhasil!


In [23]:
print("Jumlah data aktual:", len(df_akomodasi))
print("Jumlah kolom:", len(df_akomodasi.columns))
print("Jumlah desa:")
print(df_akomodasi["desa"].value_counts(dropna=False))
print("\nJumlah klasifikasi:")
print(df_akomodasi["klasifikasi"].value_counts(dropna=False))

Jumlah data aktual: 104
Jumlah kolom: 7
Jumlah desa:
desa
Desa_Ubud_2    21
Desa_Ubud_3    21
Desa_Ubud_4    21
Desa_Ubud_5    21
Desa_Ubud_1    20
Name: count, dtype: Int64

Jumlah klasifikasi:
klasifikasi
Akomodasi Lainnya    52
Non-Bintang          52
Name: count, dtype: Int64


In [32]:

from pathlib import Path

print("========== PEMERIKSAAN AKHIR UTS ==========")

for folder in [BASE_DIR]:
    for item in sorted(folder.rglob("*")):
        if item.is_file():
            ukuran_kb = item.stat().st_size / 1024
            print(f"{item.relative_to(BASE_DIR)} | {ukuran_kb:.2f} KB")

print("\n========== FILE PENTING ==========")

file_penting = [
    BASE_DIR / "README.md",
    BASE_DIR / "UTS_DAM_UBUD_RISKA_2501010315.ipynb",
    BASE_DIR / "requirements.txt",
    META_DIR / "provenance.json",
    PROCESSED_DIR / "akomodasi_cuaca_ubud.csv"
]

for file in file_penting:
    status = "ADA" if file.exists() else "BELUM ADA"
    print(status, "|", file.name)


========== PEMERIKSAAN AKHIR UTS ==========
README.md | 3.55 KB
UTS_DAM_UBUD_RISKA_2501010315.ipynb | 28.62 KB
data/interim/akomodasi_ubud_bps_2015_ekstraksi.csv | 27.46 KB
data/processed/README.md | 0.29 KB
data/processed/akomodasi_cuaca_ubud.csv | 12.21 KB
data/raw/BPS5104-direktori_hotel_kabupaten_gianyar_2017.pdf | 4365.88 KB
data/raw/README_raw.txt | 0.21 KB
docs/PANDUAN_FINALISASI.md | 2.19 KB
docs/dokumentasi.md | 1.13 KB
metadata/data_ethics_checklist.csv | 2.44 KB
metadata/data_ethics_checklist.xlsx | 7.33 KB
metadata/provenance.json | 1.26 KB
requirements.txt | 0.04 KB

========== FILE PENTING ==========
ADA | README.md
ADA | UTS_DAM_UBUD_RISKA_2501010315.ipynb
ADA | requirements.txt
ADA | provenance.json
ADA | akomodasi_cuaca_ubud.csv


In [33]:

import nbformat
from pathlib import Path

notebook_path = BASE_DIR / "UTS_DAM_UBUD_RISKA_2501010315.ipynb"

nb = nbformat.read(notebook_path, as_version=4)

jumlah_code = 0
jumlah_output = 0
jumlah_belum_dijalankan = 0

for cell in nb.cells:
    if cell.cell_type == "code":
        jumlah_code += 1
        jumlah_output += len(cell.get("outputs", []))

        if cell.get("execution_count") is None:
            jumlah_belum_dijalankan += 1

print("Jumlah cell kode:", jumlah_code)
print("Jumlah cell yang memiliki output:",
      sum(1 for c in nb.cells if c.cell_type == "code" and len(c.get("outputs", [])) > 0))
print("Jumlah output:", jumlah_output)
print("Cell belum dijalankan:", jumlah_belum_dijalankan)

print("\n=== PEMERIKSAAN CSV HASIL AKHIR ===")
import pandas as pd

hasil = pd.read_csv(PROCESSED_DIR / "akomodasi_cuaca_ubud.csv")
print("Jumlah baris:", len(hasil))
print("Jumlah kolom:", len(hasil.columns))
print("Nama kolom:", hasil.columns.tolist())
print(hasil.head())


Jumlah cell kode: 8
Jumlah cell yang memiliki output: 0
Jumlah output: 0
Cell belum dijalankan: 8

=== PEMERIKSAAN CSV HASIL AKHIR ===
Jumlah baris: 104
Jumlah kolom: 11
Nama kolom: ['no_sumber', 'desa', 'nama_akomodasi', 'alamat', 'klasifikasi', 'kecamatan', 'tahun_data', 'periode_cuaca', 'suhu_rata2_bulanan_c', 'total_curah_hujan_mm', 'angin_maks_kmh']
   no_sumber         desa      nama_akomodasi               alamat  \
0          1  Desa_Ubud_2  Hotel Ubud Indah 1  Jl. Raya Ubud No. 1   
1          2  Desa_Ubud_3  Hotel Ubud Indah 2  Jl. Raya Ubud No. 2   
2          3  Desa_Ubud_4  Hotel Ubud Indah 3  Jl. Raya Ubud No. 3   
3          4  Desa_Ubud_5  Hotel Ubud Indah 4  Jl. Raya Ubud No. 4   
4          5  Desa_Ubud_1  Hotel Ubud Indah 5  Jl. Raya Ubud No. 5   

         klasifikasi kecamatan  tahun_data periode_cuaca  \
0  Akomodasi Lainnya      Ubud        2015       2025-01   
1        Non-Bintang      Ubud        2015       2025-01   
2  Akomodasi Lainnya      Ubud        2015

/usr/local/lib/python3.13/dist-packages/nbformat/validator.py:434: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  _validate(nbdict, ref, version, version_minor, relax_add_props)


In [34]:

import os

print("Folder kerja:", os.getcwd())

try:
    from google.colab import _message
    print("Google Colab terdeteksi.")
except ImportError:
    print("Google Colab tidak terdeteksi.")


Folder kerja: /content/UTS_DAM_UBUD_RISKA_2501010315
Google Colab terdeteksi.


In [ ]:

from google.colab import _message
import json
from pathlib import Path

# Meminta data notebook aktif dari Google Colab
notebook_data = _message.blocking_request(
    "get_ipynb",
    timeout_sec=30
)

print("Tipe data:", type(notebook_data))
print("Notebook aktif berhasil diminta dari Colab.")
print("Periksa hasil di atas sebelum melanjutkan.")
